In [102]:
import os 
import openai
from langsmith import Client
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [103]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


In [104]:
client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [105]:
dataset = client.read_dataset(dataset_name="rag-evaluation-dataset")

In [106]:
list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs

{'answer': ["Among the chunks, B010 includes 'This product has good value for money. worth buying.' so B is correct."]}

In [107]:
reference_input = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].inputs

In [108]:
reference_input

{'question': "Which product ID has a review 'This product has good value for money. worth buying.'?"}

In [109]:
reference_output = list(client.list_examples(dataset_id=dataset.id, limit=10))[0].outputs
reference_output

{'answer': ["Among the chunks, B010 includes 'This product has good value for money. worth buying.' so B is correct."]}

Rag Pipeline

In [110]:

from langsmith import traceable, get_current_run_tree
import openai
from qdrant_client import QdrantClient
import os

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")


def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run and response.usage:
        # `add_metadata` is a method on the run object; call it with a dict
        try:
            current_run.add_metadata({
                "usage_metadata": {
                    "input_tokens": response.usage.prompt_tokens,
                    "total_tokens": response.usage.total_tokens,
                    "embedding_model": model,
                }
            })
        except Exception:
            # Fallback: ignore metadata errors to avoid breaking embedding
            logger = __import__("logging").getLogger(__name__)
            logger.debug("Failed to add embedding usage metadata to run")
    return response.data[0].embedding


def retrieve_data(query, qdrant_client=None, k=5):
    if qdrant_client is None:
        qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="amazon_reviews_collection",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    retrieve_context_details = []
    retrieve_context_product_names = []
    retrieve_context_helpful_votes = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(payload.get('product_id'))
        retrieve_context.append(payload.get('review_text', ''))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(payload.get('rating', 0))
        retrieve_context_helpful_votes.append(payload.get('helpful_votes', 0))
        retrieve_context_product_names.append(payload.get('product_name', ''))
        retrieve_context_details.append(payload.get('details', ''))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_ratings': retrieved_context_ratings,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_product_names': retrieve_context_product_names,
        'retrieve_context_helpful_votes': retrieve_context_helpful_votes
    }


def process_context(context):
    formatted_context = []
    for id, chunk, rating, details, product_name, helpful_votes in zip(context['retrieved_context_ids'], context['retrieve_context'], context['retrieved_context_ratings'], context['retrieve_context_details'], context['retrieve_context_product_names'], context['retrieve_context_helpful_votes']):
        formatted_context.append(f"ID: {id}\nRating: {rating}\nReview: {chunk}\nDetails: {details}\nProduct Name: {product_name}\nHelpful Votes: {helpful_votes}\n---")
    return "\n".join(formatted_context)

@traceable(
        name="build_prompt",
        tags=["prompt_construction"],
        run_type="prompt"
)
def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt

@traceable(
        name="gen_answer",
        tags=["answer_generation", "openai"],
        run_type="llm",
        metadata={"model": "gpt-5-nano", "ls-provider": "openai"}
)
def gen_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}]
    )

    current_run = get_current_run_tree()
    if current_run:
        try:
            current_run.add_metadata({
                "usage_metadata": {
                    "input_tokens": response.usage.prompt_tokens,
                    "output_tokens": response.usage.completion_tokens,
                    "total_tokens": response.usage.total_tokens,
                    "generation_model": "gpt-5-nano",
                }
            })
        except Exception:
            logger = __import__("logging").getLogger(__name__)
            logger.debug("Failed to add generation usage metadata to run")
    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = gen_answer(prompt)

    final_result = {
        "question": question,
        "answer": answer,
        "retrieved_context": retrieve_context
    }

    return final_result

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


Ragas Metrics

In [111]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_54084/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_54084/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_54084/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [112]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_54084/3421941080.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_54084/3421941080.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [113]:
reference_input

{'question': "Which product ID has a review 'This product has good value for money. worth buying.'?"}

In [114]:
reference_output

{'answer': ["Among the chunks, B010 includes 'This product has good value for money. worth buying.' so B is correct."]}

In [115]:
result = rag_pipeline(reference_input["question"])

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_54084/3814530283.py:149: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)


In [116]:
result

{'question': "Which product ID has a review 'This product has good value for money. worth buying.'?",
 'answer': 'B010 and B004.',
 'retrieved_context': {'retrieved_context_ids': ['B010',
   'B004',
   'B002',
   'B005',
   'B007'],
  'retrieve_context': ['This product has good value for money. worth buying.',
   'This product has good value for money. worth buying.',
   'This product has good value for money. great product.',
   'This product has good value for money. pretty decent.',
   'This product has good value for money. pretty decent.'],
  'similarity_scores': [0.63067615, 0.630649, 0.57734, 0.54464424, 0.54464424],
  'retrieved_context_ratings': [1, 4, 2, 4, 4],
  'retrieve_context_details': [{'Date First Available': 'May 30, 2024',
    'Manufacturer': 'Brand A',
    'ASIN': 'ASIN5643299'},
   {'Date First Available': 'April 19, 2024',
    'Manufacturer': 'Brand C',
    'ASIN': 'ASIN1873301'},
   {'Date First Available': 'March 02, 2024',
    'Manufacturer': 'Brand A',
    'AS

In [118]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]["retrieve_context"],
    )
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

In [119]:
await ragas_faithfulness(result, '')

0.0

In [120]:
async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]["retrieve_context"],
    )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)

In [121]:
await ragas_response_relevancy(result, '')

np.float64(0.326491623400524)

In [133]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context"]["retrieved_context_ids"],
        reference_context_ids=example.get("retrieved_context_ids", [])
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [134]:
await ragas_context_precision_id_based(result, reference_output)

0.0